# Baseline (FM) model inspection

Walks the whole path so the pieces are concrete:
**raw row → 5 encoded ids → embedding table `V` → the score**.

Run top to bottom. Make sure the kernel is the project `.venv`
(`.venv\Scripts\python.exe`).

In [1]:
import numpy as np
from tippytop.data.dataset import load_dataset
from tippytop.config import DATA_DIR
from tippytop.kit import FIELDS

data = load_dataset(DATA_DIR)
print("FIELDS:", FIELDS)
print("split sizes:", {k: len(v) for k, v in data.splits.items()})
print("embedding-table size dim:", data.dim)

FIELDS: ['user_id', 'video_id', 'author_id', 'tab', 'dur_bucket']
split sizes: {'train': 1141112, 'valid': 124909, 'test': 170588}
embedding-table size dim: 40260


In [6]:
data.splits['train']

[(20220411, '0', '1527', '8453239', '1', 209900.0, 0),
 (20220416, '0', '7405', '8148591', '0', 65400.0, 0),
 (20220420, '0', '6026', '7535492', '1', 170833.0, 0),
 (20220411, '1', '6354', '5344499', '8', 255160.0, 0),
 (20220411, '1', '3645', '5744165', '1', 79733.0, 0),
 (20220412, '1', '4073', '5485907', '1', 114680.0, 1),
 (20220412, '1', '1725', '7515823', '1', 156433.0, 1),
 (20220412, '1', '3891', '6182848', '1', 173800.0, 1),
 (20220412, '1', '5606', '11796', '0', 109320.0, 0),
 (20220417, '1', '4352', '2871460', '1', 12576.0, 1),
 (20220409, '2', '5109', '5570290', '0', 102666.0, 0),
 (20220409, '2', '1225', '7077142', '1', 148916.0, 0),
 (20220409, '2', '3876', '7580159', '1', 89666.0, 1),
 (20220409, '2', '6524', '8082371', '1', 119666.0, 1),
 (20220410, '2', '4595', '6149869', '1', 44750.0, 1),
 (20220410, '2', '5109', '5570290', '1', 102666.0, 1),
 (20220410, '2', '3914', '143205', '0', 102120.0, 0),
 (20220410, '2', '3706', '7673649', '0', 125416.0, 0),
 (20220410, '2', '

In [24]:
user_ids = [x[0] for x in data.splits['train']]
print(unique_user_ids := np.unique(user_ids).shape[0], "unique user ids in train split")

13 unique user ids in train split


In [21]:
# for key, elem in data.enc.items():
#     print(f"{key}: {elem}")

a,b,c = data.enc['train']

print(a.shape)
print(b.shape)
print(len(c))

(1141112, 5)
(1141112,)
1141112


## 1. A raw row

`load()` reduces each impression to a 7-tuple:
`(date, user_id, video_id, author_id, tab, duration_ms, long_view)`.

In [25]:
cols = ["date", "user_id", "video_id", "author_id", "tab", "duration_ms", "long_view"]
for x in data.splits["train"][:3]:
    print(dict(zip(cols, x)))

{'date': 20220411, 'user_id': '0', 'video_id': '1527', 'author_id': '8453239', 'tab': '1', 'duration_ms': 209900.0, 'long_view': 0}
{'date': 20220416, 'user_id': '0', 'video_id': '7405', 'author_id': '8148591', 'tab': '0', 'duration_ms': 65400.0, 'long_view': 0}
{'date': 20220420, 'user_id': '0', 'video_id': '6026', 'author_id': '7535492', 'tab': '1', 'duration_ms': 170833.0, 'long_view': 0}


## 2. The encoded row = 5 integer ids

`encode()` turns the 5 fields into 5 integers that index one shared table.
`X` has shape `(N, 5)`. These are *indices, not values*.

In [26]:
X = data.X("train")
y = data.y("train")
print("X shape:", X.shape, "| dtype:", X.dtype)
print("first raw row  :", data.splits["train"][0])
print("first encoded X:", X[0], "| label y:", y[0])

X shape: (1141112, 5) | dtype: int32
first raw row  : (20220411, '0', '1527', '8453239', '1', 209900.0, 0)
first encoded X: [    0 26211 33750 40233 40249] | label y: 0.0


## 3. Vocab & offsets — how a value becomes an id

The kit builds one vocab per field from **train only** (+1 UNK slot each), then
offsets each field into its own block of the table. We rebuild that here just to
look inside (this mirrors `data.encode`).

In [27]:
import data as kitdata  # the kit's data.py, on sys.path via tippytop.kit

tr = data.splits["train"]
edges = kitdata._bucket_edges([x[5] for x in tr])  # duration_ms train deciles

def raw(x):
    return [x[1], x[2], x[3], x[4], str(int(np.searchsorted(edges, x[5])))]

vocabs = [dict() for _ in FIELDS]
for x in tr:
    for i, v in enumerate(raw(x)):
        if v not in vocabs[i]:
            vocabs[i][v] = len(vocabs[i])
field_dims = [len(v) + 1 for v in vocabs]          # +1 = UNK slot
offsets = np.cumsum([0] + field_dims[:-1])

for name, d, off in zip(FIELDS, field_dims, offsets):
    print(f"{name:11s} vocab={d-1:6d} (+1 UNK)  block starts at V row {off}")
print("total dim:", sum(field_dims), "== data.dim:", data.dim)

user_id     vocab= 26210 (+1 UNK)  block starts at V row 0
video_id    vocab=  7538 (+1 UNK)  block starts at V row 26211
author_id   vocab=  6482 (+1 UNK)  block starts at V row 33750
tab         vocab=    15 (+1 UNK)  block starts at V row 40233
dur_bucket  vocab=    10 (+1 UNK)  block starts at V row 40249
total dim: 40260 == data.dim: 40260


### Decode the first row's 5 ids back to (field, value)

Each encoded id = `local_id + offset[field]`. Subtract the offset to recover the
per-field id, then reverse the vocab.

In [28]:
inv = [{i: v for v, i in vc.items()} for vc in vocabs]

for i, gid in enumerate(X[0]):
    local = gid - offsets[i]
    val = inv[i].get(local, "<UNK>")
    print(f"{FIELDS[i]:11s} global id {gid:6d}  =  local {local:6d}  ->  value {val!r}")

user_id     global id      0  =  local      0  ->  value '0'
video_id    global id  26211  =  local      0  ->  value '1527'
author_id   global id  33750  =  local      0  ->  value '8453239'
tab         global id  40233  =  local      0  ->  value '1'
dur_bucket  global id  40249  =  local      0  ->  value '8'


## 4. Train the FM

~50s on CPU. Early-stops on valid primary and restores the best snapshot.

In [29]:
from tippytop.models import build

model = build("fm", seed=0)
model.fit(data)
m = model._m  # the trained numpy FM core
print("\nV (embedding table):", m.V.shape, "| W (linear):", m.W.shape, "| b:", float(m.b))

  epoch  1 | loss 0.6391 | valid GAUC 0.6467 nDCG@5 0.5272 primary 0.5869 | 5.1s
  epoch  2 | loss 0.5479 | valid GAUC 0.6589 nDCG@5 0.5323 primary 0.5956 | 5.1s
  epoch  3 | loss 0.5129 | valid GAUC 0.6642 nDCG@5 0.5344 primary 0.5993 | 5.0s
  epoch  4 | loss 0.5004 | valid GAUC 0.6642 nDCG@5 0.5346 primary 0.5994 | 4.5s
  epoch  5 | loss 0.4941 | valid GAUC 0.6661 nDCG@5 0.5360 primary 0.6010 | 5.3s
  epoch  6 | loss 0.4897 | valid GAUC 0.6658 nDCG@5 0.5354 primary 0.6006 | 5.0s
  epoch  7 | loss 0.4859 | valid GAUC 0.6671 nDCG@5 0.5358 primary 0.6015 | 3.7s
  epoch  8 | loss 0.4821 | valid GAUC 0.6665 nDCG@5 0.5359 primary 0.6012 | 4.7s
  epoch  9 | loss 0.4784 | valid GAUC 0.6666 nDCG@5 0.5348 primary 0.6007 | 4.5s
  epoch 10 | loss 0.4744 | valid GAUC 0.6650 nDCG@5 0.5342 primary 0.5996 | 5.0s
  epoch 11 | loss 0.4705 | valid GAUC 0.6640 nDCG@5 0.5341 primary 0.5990 | 4.3s
  early stop at epoch 11

V (embedding table): (40260, 16) | W (linear): (40260,) | b: -0.02028670720756054


## 5. The embedding of an id = a row of `V`

No network computes it — the embedding of a given id is literally `V[id]`
(a length-16 vector). The 5 ids of a row gather 5 such rows.

In [30]:
gid_user = X[0, 0]   # the user_id id of row 0
gid_video = X[0, 1]  # the video_id id of row 0

print("user embedding  V[", gid_user, "] =\n", m.V[gid_user])
print("\nvideo embedding V[", gid_video, "] =\n", m.V[gid_video])

# E = V[X] gathers all 5 embeddings for a batch of rows:
E0 = m.V[X[0]]        # (5, 16)
print("\nE for row 0:", E0.shape, "= 5 fields x 16 dims")

user embedding  V[ 0 ] =
 [ 0.06553294 -0.10177588 -0.04341494  0.06386635 -0.02800557  0.08241326
 -0.06648287 -0.03493125  0.03928808  0.04792913  0.03395111 -0.02131849
 -0.10303179  0.03526691  0.08370779 -0.0444068 ]

video embedding V[ 26211 ] =
 [-0.19896805  0.1562398   0.01895734 -0.1262501  -0.03498711 -0.10541379
 -0.00807204  0.0297395   0.03046587 -0.0138764   0.02728665  0.0845597
  0.03775759  0.00698537 -0.00399783  0.09880966]

E for row 0: (5, 16) = 5 fields x 16 dims


## 6. Score one row by hand, and check it matches

`score = b + Σ W[id]  +  ½·[(ΣE)² − ΣE²]` (the FM trick for all pairwise
interactions). Recompute it manually and compare to `model.predict`.

In [31]:
ids = X[0]
E = m.V[ids]                     # (5, 16)
S = E.sum(0)                     # (16,)
inter = 0.5 * ((S ** 2).sum() - (E ** 2).sum())
manual = float(m.b) + m.W[ids].sum() + inter

via_model = model.predict(data, "train")[0]
print("manual score :", manual)
print("model.predict:", via_model)
print("match:", np.isclose(manual, via_model, atol=1e-4))

manual score : -0.8722571
model.predict: -0.8722571
match: True


### The user × video pairwise term (the signal that does the work)

Of all pairs, `⟨user, video⟩` is the one the docs say carries most of the
learnable signal — “this user likes this video”.

In [32]:
user_vec, video_vec = m.V[X[0, 0]], m.V[X[0, 1]]
print("<user, video> dot product:", float(user_vec @ video_vec))

# all 10 pairwise dot products among the 5 fields of row 0:
for a in range(5):
    for b_ in range(a + 1, 5):
        dp = float(m.V[X[0, a]] @ m.V[X[0, b_]])
        print(f"  <{FIELDS[a]:10s}, {FIELDS[b_]:10s}> = {dp:+.4f}")

<user, video> dot product: -0.05474717542529106
  <user_id   , video_id  > = -0.0547
  <user_id   , author_id > = -0.0573
  <user_id   , tab       > = -0.4142
  <user_id   , dur_bucket> = +0.1097
  <video_id  , author_id > = +0.0491
  <video_id  , tab       > = +0.3531
  <video_id  , dur_bucket> = -0.2265
  <author_id , tab       > = +0.3368
  <author_id , dur_bucket> = -0.2161
  <tab       , dur_bucket> = -0.2786


# EDA

In [39]:
import pandas as pd

cols = ["date", "user_id", "video_id", "author_id", "tab", "duration_ms", "long_view"]
df = pd.DataFrame(data.splits["train"], columns=cols)
df

,date,user_id,video_id,author_id,tab,duration_ms,long_view
0,20220411,0,1527,8453239,1,209900.0,0
1,20220416,0,7405,8148591,0,65400.0,0
2,20220420,0,6026,7535492,1,170833.0,0
3,20220411,1,6354,5344499,8,255160.0,0
4,20220411,1,3645,5744165,1,79733.0,0
...,...,...,...,...,...,...,...
1141107,20220417,27283,4506,6723181,1,22566.0,0
1141108,20220418,27283,2553,8670316,1,10466.0,1
1141109,20220418,27283,5919,3419364,1,95566.0,0
1141110,20220410,27284,7136,5802877,2,259066.0,0


In [41]:
df = pd.read_csv(r"D:\Coding\Hackathon\2026\TiktokTechjam\Tippytop\kuairand-starter-kit\kuairand-starter-kit\KuaiRand-Pure\data\log_random_4_22_to_5_08_pure.csv")
df

,user_id,video_id,date,hourmin,time_ms,is_click,is_like,is_follow,is_comment,is_forward,is_hate,long_view,play_time_ms,duration_ms,profile_stay_time,comment_stay_time,is_profile_enter,is_rand,tab
0,0,5543,20220430,1800,1651314030792,0,0,0,0,0,0,0,863,30066,0,0,0,1,1
1,0,1836,20220502,1200,1651466607423,0,0,0,0,0,0,0,1190,132433,0,0,0,1,1
2,0,721,20220502,1700,1651481542743,0,0,0,0,0,0,0,1528,25266,0,0,0,1,1
3,0,442,20220502,1800,1651488577163,0,0,0,0,0,0,0,1490,12000,0,0,0,1,1
4,0,62,20220503,800,1651535940163,0,0,0,0,0,0,0,1419,169500,0,0,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1186054,27284,4767,20220504,2200,1651672569530,0,0,0,0,0,0,0,1498,12600,0,0,0,1,1
1186055,27284,4181,20220504,2200,1651672699078,0,0,0,0,0,0,0,2017,17950,0,0,0,1,1
1186056,27284,3340,20220504,2200,1651672816288,0,0,0,0,0,0,0,4039,70160,0,0,0,1,1
1186057,27284,4717,20220508,1500,1651993401641,1,0,0,0,0,0,1,18311,62100,0,0,0,1,1


['user_id', 'video_id', 'author_id', 'tab', 'dur_bucket']